In [ ]:
import s3fs
import rasterio
import xarray as xr
import rioxarray
import duckdb
import numpy as np
import matplotlib.pyplot as plt
import boto3
import pandas as pd
import geopandas as gpd

## Check that treemap is in scratch bucket

In [ ]:
fs = s3fs.S3FileSystem()
fs.ls("s3://nasa-cryo-scratch/s-kganz/treemap/Data/")

## Summarize BA per treemap ID

In [ ]:
# It seems like downloading the database is the best option. Save to temp so we
# don't have extra data lying around.
fs.download(
    "nasa-cryo-scratch/s-kganz/treemap/Data/TreeMap2016_tree_table.db",
    "/tmp/TreeMap2016_tree_table.db"
)

In [ ]:
# Mountain pine beetle
host_spcodes = (113,108,122,101,116,117,119,109,102,142,114,133,104,105,106)

db = duckdb.connect("/tmp/TreeMap2016_tree_table.db")
db.sql("SHOW TABLES")

In [ ]:
db.sql("SELECT * FROM TreeMap2016_tree_table LIMIT 5").df()

In [ ]:
# This query will omit TM IDs that have no basal area (because of dead trees
# or none of the host species) ...
hostba_where_present = db.sql(
    f"""
    SELECT 
        tm_id,
        CAST(SUM(DIA * DIA * 0.005454 * TPA_UNADJ) AS INT) as host_ba
    FROM TreeMap2016_tree_table
    WHERE 
        STATUSCD == 1 AND
        SPCD IN {host_spcodes}
    GROUP BY tm_id
    """
).df()

# ... so also make a table with all possible TM IDs ...
all_hostba = db.sql(
    """
    SELECT DISTINCT tm_id FROM TreeMap2016_tree_table
    """
).df()

all_hostba["host_ba"] = 0
all_hostba = all_hostba.set_index("tm_id")

# ... and update the full table with TM IDs that 
# have trees. Idk why this warns update() is always inplace
all_hostba.host_ba.update(hostba_where_present["host_ba"])

In [ ]:
# Account for nodata pixels
nodata_tmid = 2147483647
all_hostba.loc[nodata_tmid] = [0]
all_hostba.tail()

## Open treemap from scratch bucket

In [ ]:
session = rasterio.session.AWSSession(boto3.Session(), requester_pays=True)
treemap = rioxarray.open_rasterio(
    "s3://nasa-cryo-scratch/s-kganz/treemap/Data/TreeMap2016.tif", 
    band_as_variable=True,
    chunks="auto"
)
print(treemap.rio.crs)
treemap

In [ ]:
# Figure out processing extent
usfs_regions = gpd.read_file("../data_in/usfs_region_boundaries/usfs_regions_simple.shp").to_crs(treemap.rio.crs)
usfs_regions_explode = usfs_regions.geometry.explode()
usfs_regions_explode = usfs_regions_explode[usfs_regions_explode.geometry.area > 2e11]
bounds = usfs_regions_explode.total_bounds
xmin, ymin, xmax, ymax = bounds
print(bounds)

In [ ]:
# sel() must be exactly on the beginning/end of a chunk for ease of use
# with map_blocks(). So snap to the nearest coordinate and then snap
# to the nearest chunk
x_snap = treemap.x.sel(x=[xmin, xmax], method="nearest")
y_snap = treemap.y.sel(y=[ymin, ymax], method="nearest")
x_idx = np.where(treemap.x.isin(x_snap))[0]
y_idx = np.where(treemap.y.isin(y_snap))[0]

chunksize_x = 5760
chunksize_y = 5760

print("Before snapping:", x_idx, y_idx)

x_idx[0] = int(chunksize_x * (np.floor(x_idx[0] / chunksize_x)))
x_idx[1] = int(chunksize_x * (np.ceil(x_idx[1] / chunksize_x)))
y_idx[0] = int(chunksize_y * (np.floor(y_idx[0] / chunksize_y)))
y_idx[1] = int(chunksize_y * (np.ceil(y_idx[1] / chunksize_y)))

print("After snapping:", x_idx, y_idx)

In [ ]:
treemap_clip = treemap.isel(
    x=slice(*x_idx),
    y=slice(*y_idx) 
)
treemap_clip

In [ ]:
# Assert that all chunks are the same size.
for dim in treemap_clip.chunksizes:
    sizes = np.array(treemap_clip.chunksizes[dim])
    assert (sizes == sizes[0]).all()

## Try processing a block

The goal here is to replace each pixel in TreeMap with the corresponding entry for host_ba in the above table. There are many more treemap IDs in the above table than unique values in each block, so it would be very efficient to np.where() for everything. So, the strategy is:

 - Read a block
 - Make a block of zeros of equal size
 - Determine unique TM ids in input block
 - For each id:
     - Get host BA for that id
     - Make a mask for that id: np.where(block == id) * host_ba
     - Add the mask to the output block
 - Coarsen the output block to ~900 m pixels

In [ ]:
%%time
block = treemap_clip.isel(x=slice(38_000, 40_000), y=slice(38_000, 40_000)).compute()
block_tmids = block.band_1.data.flatten()
block_ba = xr.DataArray(
    data=all_hostba.host_ba.loc[block_tmids].to_numpy().reshape(block.band_1.shape),
    dims=block.dims,
    coords=block.coords
)
block_coarse = block_ba.coarsen(dict(y=8, x=8)).mean()

## Process using map_blocks()

The flatten step pulls all the data in the chunk into memory, so the best option for processing the full array is `xr.map_blocks`.

In [ ]:
# Make a local cluster for parallelism
from dask.distributed import Client

client = Client()
client

In [ ]:
coarsen_factor = 8

def process_block(block: xr.Dataset) -> xr.DataArray:
    block_tmids = block.band_1.data.flatten()
    block_ba = xr.DataArray(
        data=all_hostba.host_ba.loc[block_tmids].to_numpy().reshape(block.band_1.shape),
        dims=block.dims,
        coords=block.coords
    )
    block_coarse = block_ba.coarsen(dict(y=coarsen_factor, x=coarsen_factor), boundary="trim").mean()
    return block_coarse

In [ ]:
template = treemap_clip.band_1.coarsen(x=coarsen_factor, y=coarsen_factor, boundary="trim").mean()

In [ ]:
mpb_ba = xr.map_blocks(
    func=process_block,
    obj=treemap_clip,
    template=template
)

In [ ]:
mpb_ba = mpb_ba.compute()

In [ ]:
mpb_ba.coarsen(x=8, y=8).mean().plot()